In [1]:
import pandas as pd
import numpy as np
import torch
import time
from tqdm import tqdm
from transformers import MarianMTModel, MarianTokenizer
from comet import download_model, load_from_checkpoint
from datasets import load_metric

/Users/choseoyeon/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/choseoyeon/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch version 2.8.0 available.


In [2]:
torch.backends.mps.is_available = lambda: False  # MacOS 대응
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
df = pd.read_csv("OpenSubtitles_en-fr_clean.csv").head(300)
print(f"Loaded {len(df)} sentences.")


Loaded 300 sentences.


In [4]:
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to("cuda" if torch.cuda.is_available() else "cpu")



/Users/choseoyeon/Library/Python/3.9/lib/python/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [5]:
def split_into_chunks(sentence, n):
    words = sentence.strip().split()
    return [" ".join(words[i:i+n]) for i in range(0, len(words), n)]


def translate_chunked_sentence(model, tokenizer, sentence, n, device="cpu"):
    """Split sentence into n-word chunks, translate each, then join."""
    chunks = split_into_chunks(sentence, n)
    translated_chunks = []
    for chunk in chunks:
        if not chunk.strip():
            continue
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True).to(device)
        outputs = model.generate(**inputs)
        translated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        translated_chunks.append(translated)
    return " ".join(translated_chunks)

In [6]:
#context_sizes = [1, 2, 3, 5]
context_sizes = [1]
results = {}

for n in context_sizes:
    preds = []
    print(f"\nTranslating with context={n} ...")

    for src in tqdm(df["src"], desc=f"context={n}"):
        pred = translate_chunked_sentence(model, tokenizer, src, n, device=device)
        preds.append(pred)

    df_context = pd.DataFrame({
        "src": df["src"],
        "tgt": df["tgt"],
        "pred": preds
    })

    output_path = f"OpenSubtitles_context{n}_translated.csv"
    df_context.to_csv(output_path, index=False, encoding="utf-8")
    print(f"Saved translated results to {output_path}")

    results[n] = df_context



Translating with context=1 ...


context=1: 100%|██████████| 300/300 [04:20<00:00,  1.15it/s]

Saved translated results to OpenSubtitles_context1_translated.csv


In [7]:
metric_bleu = load_metric("sacrebleu")
metric_chrf = load_metric("chrf")

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

/var/folders/00/hlc640bs3hx3jf_rqtdxfpkc0000gn/T/ipykernel_20001/786032477.py:1: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  metric_bleu = load_metric("sacrebleu")
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 58092.85it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/Users/choseoyeon/Library/Python/3.9/lib/python/site-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [8]:
final_scores = []

for n, df_context in results.items():
    print(f"\nEvaluating context={n} ...")

    src_texts = df_context["src"].tolist()
    tgt_texts = df_context["tgt"].tolist()
    preds = df_context["pred"].tolist()

    # --- BLEU / chrF ---
    bleu = metric_bleu.compute(predictions=preds, references=[[t] for t in tgt_texts])
    chrf = metric_chrf.compute(predictions=preds, references=[[t] for t in tgt_texts])

    # --- COMET ---
    data = [{"src": s, "mt": p, "ref": t} for s, p, t in zip(src_texts, preds, tgt_texts)]
    comet_score = comet_model.predict(data, batch_size=4, gpus=0, num_workers=0)
    comet_mean = np.mean(comet_score["system_score"])

    final_scores.append({
        "context": n,
        "BLEU": bleu["score"],
        "chrF": chrf["score"],
        "COMET": comet_mean
    })

    print(f"Context={n}: BLEU={bleu['score']:.2f}, chrF={chrf['score']:.2f}, COMET={comet_mean:.4f}")


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs



Evaluating context=1 ...


Predicting DataLoader 0: 100%|██████████| 75/75 [00:33<00:00,  2.23it/s]

Context=1: BLEU=2.05, chrF=31.32, COMET=0.4332


In [9]:
results_df = pd.DataFrame(final_scores)
results_df.to_csv("evaluation_summary.csv", index=False)
print("\n===== Final Evaluation Summary =====")
print(results_df)


===== Final Evaluation Summary =====
   context      BLEU       chrF    COMET
0        1  2.052113  31.321716  0.43318
